# Part 2: Devices, and train mode against evaluation mode

Two habits belong in every training script. The first is to write code that runs the same on a CPU or a GPU. The second is to put the model in the right **mode** for what you are doing.

In [ ]:
import torch
from torch import nn

from churn_data import make_loaders, make_splits
from churn_net import ChurnNet


def pick_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


device = pick_device()
print("training on:", device)

splits = make_splits(42)
model = ChurnNet(splits.n_features).to(device)
batch = splits.val.x[:3].to(device)
print(model(batch).device)

Move the **model** once, and move **every batch** inside the loop. The model and its inputs must live on the same device. If they do not, PyTorch raises a `RuntimeError` that names both devices. Bring results back with `.cpu()` before you hand them to NumPy or scikit-learn. When you load a checkpoint saved on a GPU onto a computer without one, add `map_location="cpu"` to `torch.load`.

## Modes

`Dropout` and `BatchNorm` behave differently while training and while predicting. `model.train()` and `model.eval()` switch them. Here is a tiny network with dropout, called twice on the same input.

In [ ]:
torch.manual_seed(0)
net = nn.Sequential(nn.Linear(9, 16), nn.ReLU(), nn.Dropout(0.5), nn.Linear(16, 1))
x = splits.val.x[:4]

net.train()
print("train mode:", net(x)[:, 0].detach().numpy().round(3), net(x)[:, 0].detach().numpy().round(3))
net.eval()
print("eval mode: ", net(x)[:, 0].detach().numpy().round(3), net(x)[:, 0].detach().numpy().round(3))

In train mode, dropout switches off half of the units at random, so the same input gives a different answer each time. In eval mode, every unit is used and the answer is stable. `BatchNorm` keeps running averages that update only in train mode:

In [ ]:
bn = nn.Sequential(nn.Linear(9, 16), nn.BatchNorm1d(16), nn.ReLU(), nn.Linear(16, 1))
print("before:", bn[1].running_mean[:3].numpy().round(3), "batches seen:", int(bn[1].num_batches_tracked))
bn.train()
bn(splits.train.x[:64])
print("after one train batch:", bn[1].running_mean[:3].detach().numpy().round(3), "batches seen:", int(bn[1].num_batches_tracked))
bn.eval()
bn(splits.train.x[64:128])
print("after one eval batch: ", bn[1].running_mean[:3].detach().numpy().round(3), "batches seen:", int(bn[1].num_batches_tracked))

## What forgetting `eval()` costs

Train a network with dropout, then measure the validation loss three ways.

In [ ]:
torch.manual_seed(1)
train_loader, val_loader, _ = make_loaders(splits)
model = nn.Sequential(
    nn.Linear(9, 32), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(32, 32), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(32, 1),
)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
for epoch in range(30):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss_fn(model(xb).squeeze(1), yb).backward()
        optimizer.step()


def validation_loss(training_mode):
    model.train(training_mode)
    losses = []
    with torch.no_grad():
        for xb, yb in val_loader:
            losses.append(loss_fn(model(xb).squeeze(1), yb).item())
    return sum(losses) / len(losses)


for _ in range(3):
    print("train mode:", round(validation_loss(True), 4))
for _ in range(2):
    print("eval mode: ", round(validation_loss(False), 4))

In train mode the number is higher and changes on every call, so you cannot compare two epochs fairly. In eval mode it is lower and repeatable.

`torch.no_grad()` is a different switch. It tells PyTorch not to record the graph, which saves memory and time. It does **not** change dropout or batch norm. When you measure, use both: `model.eval()` and `torch.no_grad()`. The `set_grad_enabled` call in Module 07's `run_epoch` does the second one, and `model.train(training)` does the first.

In [ ]:
model.eval()
big = torch.randn(200_000, 9)
print("with graph recorded:", model(big).requires_grad)
with torch.no_grad():
    print("inside no_grad:     ", model(big).requires_grad)

## Your turn

Remove `model.eval()` from a copy of your Lab 07 validation function and plot the validation curve from two runs with the same seed. How different are they?